# 2 — Modélisation finale et grille de score (`SB`)

Version de production du notebook de modélisation (segment `SB` — mono-détenus avec engagement), ne conservant que la **méthode finale retenue** (`Logit guidé + découpage fin`) et le **modèle concurrent de référence** (`XGBoost brut optimisé`) :
1. Chargement de la base modélisation, des règles de binning et découpage interne `fit` / `valid`.
2. Modèle concurrent (`XGBoost brut optimisé`, ch. 5.2) et importance par permutation.
3. Sélection initiale par ordre de liaison (`stepwise_liaison` + second passage, ch. 5.5) et regroupements initiaux.
4. Enrichissement guidé par XGBoost, découpage fin des 4 variables clés et test par modalité (ch. 5.1).
5. Estimation du modèle final (erreurs standard robustes par client) et comparaison bootstrap / ROC face au concurrent.
6. Diagnostics réglementaires complets (tests par variable, odds ratios & IC 95 %, VIF, monotonie des coefficients, stabilité).
7. Construction des grilles de score (méthode PDO et normalisée, ch. 6) et vérification de la conservation du Gini.
8. Exports de production (`regles_modele_SB.json`, `grille_SB.xlsx`, `base_score_SB.parquet`).

## 1. Paramètres, chargement et validation interne

In [ ]:
# Cellule 1 — paramètres, chargement et validation interne (20 % des clients du train)
import json
from pathlib import Path
import matplotlib.pyplot as plt
import polars as pl
from utils import binning, exploration, models, scorecard
from utils.segmentation import split_train_test

pl.Config.set_tbl_rows(-1); pl.Config.set_tbl_cols(-1); pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_hide_column_data_types(True)

ID, TARGET, DATE = "gdt", "ddefaut_ndb", "datdelhis"
CODE = "SB"
SCORE_REF, ODDS_REF, PDO = 600, 50, 20

base = pl.read_parquet(f"data/processed/base_modele_{CODE}.parquet")
regles = binning.charger_regles(f"outputs/scorecard/regles_{CODE}.json")
selection = pl.read_excel(f"outputs/binning_{CODE}.xlsx", sheet_name=f"{CODE}_Selection")
VARS_MODELE = selection.filter(pl.col("statut") == "retenue")["variable"].to_list()

train, test = base.filter(pl.col("echantillon") == "train"), base.filter(pl.col("echantillon") == "test")
train_v = split_train_test(train.drop("echantillon"), ID, TARGET, [DATE], part_test=0.2, seed=1)
fit, valid = train_v.filter(pl.col("echantillon") == "train"), train_v.filter(pl.col("echantillon") == "test")
print(f"{len(VARS_MODELE)} variables | train : {train.height:,} (fit {fit.height:,} / valid {valid.height:,}) | test : {test.height:,}")

## 2. Modèle concurrent (`XGBoost brut optimisé`) et importance par permutation

In [ ]:
# Cellule 2 — modèle concurrent (XGBoost brut optimisé) et importance par permutation sur le test
X_fit, X_valid, X_train_brut, X_test_brut = (
    models.matrice_xgb(d, VARS_MODELE, regles, TARGET, "brut", ref=train)
    for d in (fit, valid, train, test)
)

RECALCULER_XGB = False
CHEMIN_XGB = Path(f"outputs/scorecard/params_xgb_{CODE}.json")
if CHEMIN_XGB.exists() and not RECALCULER_XGB:
    PARAMS_XGB = json.loads(CHEMIN_XGB.read_text())["params"]
else:
    PARAMS_XGB, essais_xgb = models.optimiser_xgb(X_train_brut, train[TARGET], train[ID], n_essais=30, n_plis=3)
    CHEMIN_XGB.write_text(json.dumps({"params": PARAMS_XGB, "essais": essais_xgb.to_dicts()}, indent=1, default=float))

xgb_opt = models.entrainer_xgb(X_fit, fit[TARGET], X_valid, valid[TARGET], **PARAMS_XGB)
pred_xgb_test = xgb_opt.predict_proba(X_test_brut)[:, 1]
print(pl.DataFrame([
    models.metriques(train[TARGET], xgb_opt.predict_proba(X_train_brut)[:, 1], "XGBoost brut optimisé — train"),
    models.metriques(test[TARGET], pred_xgb_test, "XGBoost brut optimisé — test"),
]).select("echantillon", "auc", "gini", "ks"))

pred_xgb = lambda d: xgb_opt.predict_proba(models.matrice_xgb(d, VARS_MODELE, regles, TARGET, "brut", ref=train))[:, 1]
perm_xgb = models.importance_permutation(test, VARS_MODELE, TARGET, pred_xgb)
print(perm_xgb.head(15))

## 3. Sélection pas à pas initiale par ordre de liaison (5.5)

In [ ]:
# Cellule 3 — stepwise par ordre de liaison (2 passages) et traitement initial des modalités
ORDRE = selection.filter(pl.col("statut") == "retenue").sort("iv_test", descending=True)["variable"].to_list()
VARS_INIT, journal_1 = models.stepwise_liaison(fit, valid, ORDRE, regles, TARGET, alpha=0.05, gain_gini_min=0.002)
ECARTEES = [v for v in ORDRE if v not in VARS_INIT]
VARS_INIT, journal_2 = models.reexaminer_ecartees(
    fit, valid, VARS_INIT, ECARTEES, regles, TARGET, alpha=0.05, gain_gini_min=0.002
)

# Regroupements métier/statistiques initiaux + rattachement prudent des manquants absents
BORNES_A_RETIRER = {
    "RATIO_ECR": [0.5],
    "TX_UTIL_DAV": [1.0],
    "horsbilan": [197.75],
    "solde_crediteur": [30000.0],
}
regles_init = regles | {v: models.retirer_bornes(regles[v], b) for v, b in BORNES_A_RETIRER.items()}
regles_init, MISSING_RATTACHES = models.rattacher_missing_absent(
    {v: regles_init[v] for v in VARS_INIT}, train, TARGET
)
print(f"Socle initial : {len(VARS_INIT)} variables | manquants rattachés : {MISSING_RATTACHES}")

## 4. Méthode finale retenue : sélection guidée et découpage fin

In [ ]:
# Cellule 4 — enrichissement guidé par XGBoost, découpage fin des 4 variables clés et test par modalité (5.1)
HORS_LOGIT = perm_xgb.filter(~pl.col("variable").is_in(VARS_INIT) & (pl.col("perte_gini") > 0))["variable"].to_list()
VARS_GUIDE, regles_guide, journal_guide = models.stepwise_guide(
    fit, valid, VARS_INIT, HORS_LOGIT, regles | regles_init, TARGET, alpha=0.05, gain_gini_min=0.001
)

FINES = perm_xgb["variable"].head(4).to_list()
regles_fines = {
    v: binning.arrondir_bornes(
        binning.regle_par_methode(train, v, TARGET, regles[v].get("methode", "quantiles").split(" ")[0], DATE, part_min=0.03)
    )
    for v in FINES
}
VARS_LOGIT, regles_modele, journal_fin = models.stepwise_guide(
    fit, valid, [v for v in VARS_GUIDE if v not in FINES], FINES, regles_guide | regles_fines, TARGET,
    alpha=0.05, gain_gini_min=0.001
)

regles_modele, journal_mod = models.regrouper_non_significatives(train, VARS_LOGIT, regles_modele, TARGET)
print(f"Modèle final : {len(VARS_LOGIT)} variables -> {VARS_LOGIT}")
if journal_mod.height:
    print("Regroupements par modalité :"); print(journal_mod)

## 5. Estimation du modèle final et comparaison face au concurrent

In [ ]:
# Cellule 5 — estimation sur tout le train (erreurs standard robustes par client) et comparaison vs XGBoost
f = models.logit_final(train, test, VARS_LOGIT, regles_modele, TARGET, ID)
logit, X_train_l, X_test_l = f["res"], f["X_train"], f["X_test"]
regles_modele, ORDRES, GROUPES, NOMS = f["regles"], f["ordres"], f["groupes"], f["noms"]

pd_train_l, pd_test_l = models.predire(logit, X_train_l), models.predire(logit, X_test_l)
print(pl.DataFrame([
    models.metriques(train[TARGET], pd_train_l, "Régression logistique finale — train"),
    models.metriques(test[TARGET], pd_test_l, "Régression logistique finale — test"),
    models.metriques(test[TARGET], pred_xgb_test, "XGBoost brut optimisé — test"),
]).select("echantillon", "auc", "gini", "ks"))

SCORES = {"Régression logistique finale": pd_test_l, "XGBoost brut optimisé": pred_xgb_test}
gini_ic, ecarts_ic = models.gini_bootstrap(test[TARGET], SCORES, test[ID], reference="Régression logistique finale", n_tirages=1000)
print(gini_ic); print(ecarts_ic)
models.courbes_roc(test[TARGET], SCORES); plt.show()

## 6. Diagnostics réglementaires (5.1)

In [ ]:
# Cellule 6 — diagnostics du modèle final
print("=== Tests par variable (rapport de vraisemblance)")
print(models.tests_par_variable(logit, X_train_l, train[TARGET], GROUPES))

odds = models.odds_ratios(logit, NOMS)
print("=== Odds ratios et IC 95 %"); print(odds)
print("Modalités non significatives (p > 5 %) :",
      odds.filter((pl.col("p_value") > 0.05) & (pl.col("terme") != "constante"))["terme"].to_list())
print("Coefficients monotones / cohérents :", f["coherent"])
print("=== VIF des indicatrices (top 5)"); print(models.vif_indicatrices(X_train_l, NOMS).head(5))

for v in FINES:
    if v in VARS_LOGIT:
        print(f"{v} | PSI mensuel max : {binning.psi_mensuel(train, v, regles_modele[v], DATE):.4f}")

## 7. Grille de score PDO et normalisée (chapitre 6)

In [ ]:
# Cellule 7 — construction de la grille PDO (et normalisée) et contrôle sur le test
FACTOR, OFFSET = scorecard.parametres_pdo(SCORE_REF, ODDS_REF, PDO)
print(f"Factor = {FACTOR:.2f} | Offset = {OFFSET:.2f}")

grille = scorecard.grille_pdo(logit, GROUPES, ORDRES, SCORE_REF, ODDS_REF, PDO)
grille_norm = scorecard.grille_normalisee(logit, GROUPES, ORDRES, total=1000)
grilles = grille.join(
    grille_norm.select("variable", "classe", pl.col("points").alias("points_normalises")),
    on=["variable", "classe"]
)
print(grilles)

score_train = scorecard.appliquer_grille(train, grille, regles_modele)
score_test = scorecard.appliquer_grille(test, grille, regles_modele)
score_test_norm = scorecard.appliquer_grille(test, grille_norm, regles_modele)

print(pl.DataFrame([
    models.metriques(test[TARGET], pd_test_l, "Régression logistique (probabilité)"),
    models.metriques(test[TARGET], -score_test, "Grille PDO (points arrondis)"),
    models.metriques(test[TARGET], -score_test_norm, "Grille normalisée (points arrondis)"),
]).select("echantillon", "gini", "ks"))
print(f"Score PDO (train) : de {score_train.min():.0f} à {score_train.max():.0f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(score_test.filter(test[TARGET] == 0), bins=40, alpha=0.6, density=True, label="sains")
ax.hist(score_test.filter(test[TARGET] == 1), bins=40, alpha=0.6, density=True, label="défauts")
ax.set(xlabel="score PDO", ylabel="densité", title=f"Distribution des scores {CODE} — test"); ax.legend(); plt.show()

print(scorecard.taux_par_tranche(score_test, test[TARGET], n_tranches=10))

## 8. Exports de production

In [ ]:
# Cellule 8 — export des artefacts du modèle final et de la base scorée pour le notebook 3 (CHR)
binning.sauver_regles(regles_modele, f"outputs/scorecard/regles_modele_{CODE}.json")
exploration.exporter_excel({f"{CODE}_Grille": grilles}, f"outputs/scorecard/grille_{CODE}.xlsx")

base_score = base.select(ID, DATE, TARGET, "echantillon").with_columns(
    segment=pl.lit(CODE),
    score=scorecard.appliquer_grille(base, grille, regles_modele),
)
base_score = base_score.with_columns(
    pd_score=pl.Series(scorecard.score_vers_pd(base_score["score"], SCORE_REF, ODDS_REF, PDO))
)
base_score.write_parquet(f"data/processed/base_score_{CODE}.parquet")
print(base_score.group_by("echantillon").agg(pl.len(), pl.col("score").mean(), pl.col("pd_score").mean()))